<a href="https://colab.research.google.com/github/TasminaTasin75/COMP6012-Assignment3/blob/main/assignment3_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from google.colab import drive
drive.mount("/content/drive")

import os

PROJECT_DIR = "/content/drive/MyDrive/COMP6012_Assignment3"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
LOGS_DIR = os.path.join(PROJECT_DIR, "logs")

for directory in [PROJECT_DIR, CHECKPOINT_DIR, RESULTS_DIR, LOGS_DIR]:
    os.makedirs(directory, exist_ok=True)

print("Google Drive connected.")
print("Project directory:", PROJECT_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)
print("Logs directory:", LOGS_DIR)

Mounted at /content/drive
Google Drive connected.
Project directory: /content/drive/MyDrive/COMP6012_Assignment3
Checkpoint directory: /content/drive/MyDrive/COMP6012_Assignment3/checkpoints
Results directory: /content/drive/MyDrive/COMP6012_Assignment3/results
Logs directory: /content/drive/MyDrive/COMP6012_Assignment3/logs


In [2]:
%pip install -q transformers datasets peft accelerate evaluate scikit-learn matplotlib pandas sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


Inspecting AG News Dataset

In [3]:
from datasets import load_dataset

ag_news_ds = load_dataset("fancyzhx/ag_news")

print("Loaded AG News Dataset:")
print(ag_news_ds)

print("\nAvailable splits:")
for split_name, split_data in ag_news_ds.items():
    print(f"{split_name}: {len(split_data):,} rows")
    print("Columns:", split_data.column_names)
    print("Features:", split_data.features)

print("\nFirst training example:")
print(ag_news_ds["train"][0])

print("\nClass labels:")
print(ag_news_ds["train"].features["label"].names)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Loaded AG News Dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Available splits:
train: 120,000 rows
Columns: ['text', 'label']
Features: {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}
test: 7,600 rows
Columns: ['text', 'label']
Features: {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}

First training example:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

Class labels:
['World', 'Sports', 'Business', 'Sci/Tech']


In [4]:

from collections import Counter

train_labels = Counter(ag_news_ds["train"]["label"])

print("AG News training examples:", f'{len(ag_news_ds["train"]):,}')
print("Training examples per label:")

label_names = ag_news_ds["train"].features["label"].names
for label_id, count in sorted(train_labels.items()):
    print(f"{label_id} — {label_names[label_id]}: {count:,}")

print("\nValidation/test split sizes:")
for split_name in ag_news_ds.keys():
    if split_name != "train":
        print(f"{split_name}: {len(ag_news_ds[split_name]):,}")

AG News training examples: 120,000
Training examples per label:
0 — World: 30,000
1 — Sports: 30,000
2 — Business: 30,000
3 — Sci/Tech: 30,000

Validation/test split sizes:
test: 7,600


**AG News Classification**
(a) Preparing train, test data

In [5]:

from datasets import DatasetDict

# Using the full official training and test splits.
ag_news_prepared_ds = DatasetDict({
    "train": ag_news_ds["train"],
    "test": ag_news_ds["test"]
})

print(ag_news_prepared_ds)

print("\nFinal split sizes:")
for split_name, split_data in ag_news_prepared_ds.items():
    print(f"{split_name}: {len(split_data):,}")

print("\nLabel names:")
print(ag_news_prepared_ds["train"].features["label"].names)

print("\nTraining label distribution:")
print(ag_news_prepared_ds["train"].unique("label"))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Final split sizes:
train: 120,000
test: 7,600

Label names:
['World', 'Sports', 'Business', 'Sci/Tech']

Training label distribution:
[2, 3, 1, 0]


Loading Qwen for classification  

In [10]:
import torch
import peft
import transformers
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    torch_dtype=torch.float16,
    ignore_mismatched_sizes=True,
)

print("Model loaded:", type(qwen_model).__name__)
print("Number of labels:", qwen_model.config.num_labels)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: Qwen2ForSequenceClassification
Number of labels: 4


Moving Qwen to GPU

In [11]:

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Stop here and check the Colab runtime.")

qwen_model = qwen_model.to("cuda") #Qwen moving to GPU here
qwen_model.eval()

print("Model device:", next(qwen_model.parameters()).device)
print(f"GPU allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"GPU total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Model device: cuda:0
GPU allocated: 1.86 GB
GPU reserved: 1.94 GB
GPU total: 14.56 GB


Attaching LoRA and inspect trainable parameters

In [21]:

from peft import LoraConfig, TaskType, get_peft_model
RANK = 8
LORA_ALPHA = 16
DROPOUT = 0.1
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=DROPOUT,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    modules_to_save=["score"],
)

qwen_lora_model = get_peft_model(qwen_model, lora_config)

qwen_lora_model.print_trainable_parameters()


trainable params: 544,256 || all params: 494,580,608 || trainable%: 0.1100


In [22]:

from transformers import AutoTokenizer
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

MAX_LENGTH = 256

def tokenize_ag_news(examples):
    return qwen_tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

qwen_train_ds = ag_news_prepared_ds["train"].map(
    tokenize_ag_news,
    batched=True,
    desc="Tokenizing full AG News training set",
)

qwen_test_ds = ag_news_prepared_ds["test"].map(
    tokenize_ag_news,
    batched=True,
    desc="Tokenizing full AG News test set",
)

In [23]:

# Keeping the model inputs and labels only.
qwen_train_model = qwen_train_ds.select_columns(
    ["input_ids", "attention_mask", "label"]
)

qwen_test_model = qwen_test_ds.select_columns(
    ["input_ids", "attention_mask", "label"]
)

# Rename labels for Hugging Face model training.
qwen_train_model = qwen_train_model.rename_column(
    "label", "labels"
)

qwen_test_model = qwen_test_model.rename_column(
    "label", "labels"
)

# Return PyTorch tensors when retrieving examples.
qwen_train_model.set_format("torch")
qwen_test_model.set_format("torch")

In [24]:
from torch.utils.data import DataLoader
import torch

BATCH_SIZE = 8

train_loader = DataLoader(
    qwen_train_model,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    qwen_test_model,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)


Creating a checkpoint folder and save the initial adapter

In [25]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

INITIAL_ADAPTER_DIR = os.path.join(CHECKPOINT_DIR, "initial_adapter")
qwen_lora_model.save_pretrained(INITIAL_ADAPTER_DIR)
qwen_tokenizer.save_pretrained(INITIAL_ADAPTER_DIR)

print("Initial adapter saved to:", INITIAL_ADAPTER_DIR)
print("Files:", os.listdir(INITIAL_ADAPTER_DIR))

Initial adapter saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/initial_adapter
Files: ['tokenizer.json', 'README.md', 'chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer_config.json']


In [27]:
qwen_lora_model.config.pad_token_id = qwen_tokenizer.pad_token_id

print("Model pad token ID:", qwen_lora_model.config.pad_token_id)
print("Tokenizer pad token:", qwen_tokenizer.pad_token)
print("Tokenizer pad token ID:", qwen_tokenizer.pad_token_id)
print("Model pad token ID:", qwen_lora_model.config.pad_token_id)

Model pad token ID: 151643
Tokenizer pad token: <|endoftext|>
Tokenizer pad token ID: 151643
Model pad token ID: 151643


Evaluating the baseline results of the model before training

In [28]:

import os
import json
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

qwen_lora_model.eval()

all_predictions = []
all_labels = []

with torch.inference_mode():
    for batch in tqdm(test_loader, desc="Qwen baseline evaluation"):
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")
        labels = batch["labels"].to("cuda")

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = qwen_lora_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        predictions = outputs.logits.argmax(dim=-1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

baseline_accuracy = accuracy_score(all_labels, all_predictions)
baseline_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro", zero_division=0
)

print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Baseline macro-F1: {baseline_macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    digits=4,
    zero_division=0
))

baseline_results = {
    "model": "Qwen2.5-0.5B-Instruct with untrained classification head",
    "evaluation": "untrained baseline",
    "test_samples": len(all_labels),
    "accuracy": baseline_accuracy,
    "macro_f1": baseline_macro_f1
}

results_path = os.path.join(CHECKPOINT_DIR, "qwen_baseline_results.json")
with open(results_path, "w") as f:
    json.dump(baseline_results, f, indent=2)

print("Baseline metrics saved to:", results_path)

Qwen baseline evaluation:   0%|          | 0/950 [00:00<?, ?it/s]

Baseline accuracy: 0.2384
Baseline macro-F1: 0.1070

Classification report:
              precision    recall  f1-score   support

       World     0.2509    0.9363    0.3958      1900
      Sports     0.0213    0.0042    0.0070      1900
    Business     0.3548    0.0058    0.0114      1900
    Sci/Tech     0.1359    0.0074    0.0140      1900

    accuracy                         0.2384      7600
   macro avg     0.1907    0.2384    0.1070      7600
weighted avg     0.1907    0.2384    0.1070      7600

Baseline metrics saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_baseline_results.json


Fine-Tune Qwen with LoRA on the Full AG News Dataset

In [30]:

import os
import json
import time
import random
import numpy as np
import torch

from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from tqdm.auto import tqdm
from safetensors.torch import load_file
from peft.utils.save_and_load import set_peft_model_state_dict


# Setting Varriables

RUN_NAME = "qwen_lora_run1"
LEARNING_RATE = 1e-5
NUM_EPOCHS = 1
CHECKPOINT_EVERY = 500
MAX_GRAD_NORM = 1.0
SEED = 42

DEVICE = "cuda"

RUN_DIR = os.path.join(CHECKPOINT_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

CHECKPOINTS_DIR = os.path.join(RUN_DIR, "checkpoints")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

POINTER_FILE = os.path.join(RUN_DIR, "latest_checkpoint.json")

assert torch.cuda.is_available(), "GPU is not available."
assert "qwen_lora_model" in globals()
assert "qwen_train_model_ds" in globals()
assert "train_loader" in globals()

BATCH_SIZE = train_loader.batch_size or 8
STEPS_PER_EPOCH = (len(qwen_train_model) + BATCH_SIZE - 1) // BATCH_SIZE
TOTAL_STEPS = STEPS_PER_EPOCH * NUM_EPOCHS

print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(qwen_train_model))
print("Batch size:", BATCH_SIZE)
print("Steps per epoch:", STEPS_PER_EPOCH)
print("Learning rate:", LEARNING_RATE)
print("Checkpoint folder:", RUN_DIR)


# Preparing the model and optimizer

qwen_lora_model.to(DEVICE)

for p in qwen_lora_model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()

trainable_params = [
    p for p in qwen_lora_model.parameters() if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")


#  Restoring progress if a checkpoint exists

global_step = 0
start_epoch = 0
batches_done = 0
loss_history = []
elapsed_before = 0.0
resumed = False

if os.path.exists(POINTER_FILE):
    with open(POINTER_FILE) as f:
        pointer = json.load(f)

    saved_dir = pointer["checkpoint_dir"]
    state_file = os.path.join(saved_dir, "training_state.pt")
    adapter_file = os.path.join(saved_dir, "adapter_model.safetensors")

    if os.path.exists(state_file) and os.path.exists(adapter_file):
        state = torch.load(
            state_file, map_location="cpu", weights_only=False
        )

        adapter_weights = load_file(adapter_file)
        set_peft_model_state_dict(
            qwen_lora_model,
            adapter_weights,
            adapter_name="default"
        )

        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])

        global_step = state["global_step"]
        start_epoch = state["next_epoch"]
        batches_done = state["batches_done"]
        loss_history = state["loss_history"]
        elapsed_before = state["elapsed_seconds"]

        # Restore random states for reproducibility.
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state_all(state["cuda_rng"])

        resumed = True
        print("Resuming from step:", global_step)
        print("Epoch index:", start_epoch)
        print("Batches already completed in this epoch:", batches_done)
    else:
        print("Checkpoint is incomplete; starting from the initial model.")

if not resumed:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# Saving a recoverable checkpoint

session_start = time.time()

def save_checkpoint(next_epoch, next_batch):
    checkpoint_name = (
        f"step_{global_step:06d}_"
        f"epoch_{next_epoch:02d}_batch_{next_batch:05d}"
    )
    checkpoint_dir = os.path.join(CHECKPOINTS_DIR, checkpoint_name)
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Save the LoRA adapter and classification head.
    qwen_lora_model.save_pretrained(checkpoint_dir)
    qwen_tokenizer.save_pretrained(checkpoint_dir)

    state = {
        "global_step": global_step,
        "next_epoch": next_epoch,
        "batches_done": next_batch,
        "loss_history": loss_history,
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "elapsed_seconds": elapsed_before + time.time() - session_start,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
    }

    torch.save(state, os.path.join(checkpoint_dir, "training_state.pt"))

    # Update the pointer only after saving the checkpoint files.
    temp_file = POINTER_FILE + ".tmp"
    with open(temp_file, "w") as f:
        json.dump({"checkpoint_dir": checkpoint_dir}, f)
    os.replace(temp_file, POINTER_FILE)

    print(f"\nCheckpoint saved: step {global_step}")


# Model Training

qwen_lora_model.train()

for epoch in range(start_epoch, NUM_EPOCHS):

    # Generate the same shuffled order when resuming this epoch.
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)

    shuffled_indices = torch.randperm(
        len(qwen_train_model),
        generator=generator
    ).tolist()

    ordered_dataset = Subset(qwen_train_model, shuffled_indices)

    # Reuse the original collator so batch formatting stays consistent.
    epoch_loader = DataLoader(
        ordered_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=train_loader.collate_fn
    )

    skip = batches_done if epoch == start_epoch else 0

    progress = tqdm(
        enumerate(epoch_loader, start=1),
        total=len(epoch_loader),
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    )

    for batch_number, batch in progress:

        # Skip batches already processed before the interruption.
        if batch_number <= skip:
            continue

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = qwen_lora_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        if not torch.isfinite(loss):
            print("\nNon-finite loss. Saving the previous recovery point.")
            save_checkpoint(epoch, batch_number - 1)
            raise RuntimeError("Training stopped because loss was non-finite.")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        # Clip large gradients to improve training stability.
        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            MAX_GRAD_NORM
        )

        # GradScaler skips the optimizer update if it detects overflow.
        scaler.step(optimizer)
        scaler.update()

        global_step += 1
        loss_value = loss.detach().float().item()
        loss_history.append(loss_value)

        recent_losses = loss_history[-100:]
        progress.set_postfix(
            loss=f"{loss_value:.4f}",
            avg100=f"{np.mean(recent_losses):.4f}"
        )

        if global_step % 100 == 0:
            progress.write(
                f"Step {global_step}/{TOTAL_STEPS} "
                f"| loss={loss_value:.4f}"
            )

        if global_step % CHECKPOINT_EVERY == 0:
            save_checkpoint(epoch, batch_number)

    # Save progress at the end of the epoch.
    save_checkpoint(epoch + 1, 0)
    batches_done = 0


# Saving the final summary
qwen_lora_model.eval()

summary = {
    "run_name": RUN_NAME,
    "epochs_requested": NUM_EPOCHS,
    "optimizer_steps_completed": global_step,
    "total_steps_expected": TOTAL_STEPS,
    "training_complete": global_step >= TOTAL_STEPS,
    "training_samples": len(qwen_train_model),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "first_100_step_mean_loss": (
        float(np.mean(loss_history[:100])) if loss_history else None
    ),
    "last_100_step_mean_loss": (
        float(np.mean(loss_history[-100:])) if loss_history else None
    ),
    "total_elapsed_seconds": elapsed_before + time.time() - session_start,
}

with open(os.path.join(RUN_DIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\nTraining summary:")
print(json.dumps(summary, indent=2))

GPU: Tesla T4
Training examples: 120000
Batch size: 8
Steps per epoch: 15000
Learning rate: 1e-05
Checkpoint folder: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_run1
Resuming from step: 15000
Epoch index: 1
Batches already completed in this epoch: 0

Training summary:
{
  "run_name": "qwen_lora_run1",
  "epochs_requested": 1,
  "optimizer_steps_completed": 15000,
  "total_steps_expected": 15000,
  "training_complete": true,
  "training_samples": 120000,
  "batch_size": 8,
  "learning_rate": 1e-05,
  "first_100_step_mean_loss": 5.198216907978058,
  "last_100_step_mean_loss": 0.3413862507045269,
  "total_elapsed_seconds": 4596.57471370697
}


Evaluating the trained Qwen model

In [31]:

import os
import json
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Use the trained model for evaluation
qwen_lora_model.eval()

all_predictions = []
all_labels = []

total_test_loss = 0.0
total_examples = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Qwen"):

        # Move data to GPU
        batch = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        # Get predictions
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = qwen_lora_model(**batch)

        # Calculate test loss
        batch_size = batch["labels"].size(0)
        total_test_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

        # Select the category with the highest score
        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

# Calculate evaluation metrics
test_accuracy = accuracy_score(all_labels, all_predictions)
test_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)
average_test_loss = total_test_loss / total_examples

class_names = ["World", "Sports", "Business", "Sci/Tech"]

print("\n===== QWEN TEST RESULTS =====")
print(f"Test examples: {total_examples}")
print(f"Test loss: {average_test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1: {test_macro_f1:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("===== CONFUSION MATRIX =====")
print(confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
))

# Save evaluation results to Google Drive
results_dir = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA/qwen_lora_run1"
)
os.makedirs(results_dir, exist_ok=True)

results = {
    "model": "Qwen2.5-0.5B-Instruct with LoRA",
    "test_examples": int(total_examples),
    "test_loss": float(average_test_loss),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "classification_report": classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        output_dict=True,
        zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3]
    ).tolist()
}

results_path = os.path.join(
    results_dir, "qwen_test_results.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {results_path}")

Evaluating Qwen:   0%|          | 0/950 [00:00<?, ?it/s]


===== QWEN TEST RESULTS =====
Test examples: 7600
Test loss: 0.2940
Accuracy: 0.9251 (92.51%)
Macro-F1: 0.9251

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

       World     0.9490    0.9211    0.9348      1900
      Sports     0.9740    0.9863    0.9801      1900
    Business     0.8895    0.8858    0.8877      1900
    Sci/Tech     0.8887    0.9074    0.8979      1900

    accuracy                         0.9251      7600
   macro avg     0.9253    0.9251    0.9251      7600
weighted avg     0.9253    0.9251    0.9251      7600

===== CONFUSION MATRIX =====
[[1750   33   69   48]
 [  12 1874    9    5]
 [  44   10 1683  163]
 [  38    7  131 1724]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_run1/qwen_test_results.json


Trial 2 Qwen: Hyperparameter Change - LR 2e-5, batch 8, 1 epoch

In [43]:

import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

BASE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA"
)
INITIAL_ADAPTER_DIR = os.path.join(BASE_DIR, "initial_adapter")

assert torch.cuda.is_available(), "GPU is not available."
assert os.path.isdir(INITIAL_ADAPTER_DIR), (
    "Initial adapter folder was not found."
)

trial2_tokenizer = AutoTokenizer.from_pretrained(
    INITIAL_ADAPTER_DIR
)

trial2_base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=4,
    torch_dtype=torch.float32,
    device_map="cpu"
)

trial2_base_model.config.pad_token_id = trial2_tokenizer.pad_token_id

trial2_model = PeftModel.from_pretrained(
    trial2_base_model,
    INITIAL_ADAPTER_DIR,
    is_trainable=True
)

trial2_model.config.pad_token_id = trial2_tokenizer.pad_token_id
trial2_model.to("cuda")

print("Fresh Trial 2 model loaded.")
print("Device:", next(trial2_model.parameters()).device)
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in trial2_model.parameters() if p.requires_grad):,}"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh Trial 2 model loaded.
Device: cuda:0
Trainable parameters: 544,256


Trail 2 : Qwen Model training

In [44]:

import os
import json
import time
import random
import numpy as np
import torch

from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from tqdm.auto import tqdm
from safetensors.torch import load_file
from peft.utils.save_and_load import set_peft_model_state_dict


# Setting Variables

RUN_NAME = "qwen_lora_trial2"
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
CHECKPOINT_EVERY = 500
MAX_GRAD_NORM = 1.0
SEED = 42

DEVICE = "cuda"

RUN_DIR = os.path.join(CHECKPOINT_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

CHECKPOINTS_DIR = os.path.join(RUN_DIR, "checkpoints")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

POINTER_FILE = os.path.join(RUN_DIR, "latest_checkpoint.json")

assert torch.cuda.is_available(), "GPU is not available."
assert "trial2_model" in globals()
assert "qwen_train_model_ds" in globals()
assert "train_loader" in globals()
assert len(qwen_train_model) == 120000

BATCH_SIZE = train_loader.batch_size or 8
STEPS_PER_EPOCH = (
    len(qwen_train_model) + BATCH_SIZE - 1
) // BATCH_SIZE
TOTAL_STEPS = STEPS_PER_EPOCH * NUM_EPOCHS

print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(qwen_train_model))
print("Batch size:", BATCH_SIZE)
print("Steps per epoch:", STEPS_PER_EPOCH)
print("Learning rate:", LEARNING_RATE)
print("Checkpoint folder:", RUN_DIR)


# Preparing the model and optimizer

trial2_model.to(DEVICE)

for p in trial2_model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()

trainable_params = [
    p for p in trial2_model.parameters() if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")


# Restoring progress if a checkpoint exists

global_step = 0
start_epoch = 0
batches_done = 0
loss_history = []
elapsed_before = 0.0
resumed = False

if os.path.exists(POINTER_FILE):
    with open(POINTER_FILE) as f:
        pointer = json.load(f)

    saved_dir = pointer["checkpoint_dir"]
    state_file = os.path.join(saved_dir, "training_state.pt")
    adapter_file = os.path.join(saved_dir, "adapter_model.safetensors")

    if os.path.exists(state_file) and os.path.exists(adapter_file):
        state = torch.load(
            state_file, map_location="cpu", weights_only=False
        )

        adapter_weights = load_file(adapter_file)
        set_peft_model_state_dict(
            trial2_model,
            adapter_weights,
            adapter_name="default"
        )

        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])

        global_step = state["global_step"]
        start_epoch = state["next_epoch"]
        batches_done = state["batches_done"]
        loss_history = state["loss_history"]
        elapsed_before = state["elapsed_seconds"]

        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state_all(state["cuda_rng"])

        resumed = True
        print("Resuming from step:", global_step)
        print("Epoch index:", start_epoch)
        print("Batches already completed in this epoch:", batches_done)
    else:
        raise RuntimeError(
            "Trial 2 checkpoint is incomplete. Inspect it before proceeding."
        )

if not resumed:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# Saving a recoverable checkpoint

session_start = time.time()

def save_checkpoint(next_epoch, next_batch):
    checkpoint_name = (
        f"step_{global_step:06d}_"
        f"epoch_{next_epoch:02d}_batch_{next_batch:05d}"
    )
    checkpoint_dir = os.path.join(CHECKPOINTS_DIR, checkpoint_name)
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Save the LoRA adapter and classification head.
    trial2_model.save_pretrained(checkpoint_dir)
    trial2_tokenizer.save_pretrained(checkpoint_dir)

    state = {
        "global_step": global_step,
        "next_epoch": next_epoch,
        "batches_done": next_batch,
        "loss_history": loss_history,
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "elapsed_seconds": elapsed_before + time.time() - session_start,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
    }

    torch.save(state, os.path.join(checkpoint_dir, "training_state.pt"))

    # Update the pointer only after saving the checkpoint files.
    temp_file = POINTER_FILE + ".tmp"
    with open(temp_file, "w") as f:
        json.dump({"checkpoint_dir": checkpoint_dir}, f)
    os.replace(temp_file, POINTER_FILE)

    print(f"\nCheckpoint saved: step {global_step}")


# Model Training

trial2_model.train()

for epoch in range(start_epoch, NUM_EPOCHS):

    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)

    shuffled_indices = torch.randperm(
        len(qwen_train_model),
        generator=generator
    ).tolist()

    ordered_dataset = Subset(qwen_train_model, shuffled_indices)

    epoch_loader = DataLoader(
        ordered_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=train_loader.collate_fn
    )

    skip = batches_done if epoch == start_epoch else 0

    progress = tqdm(
        enumerate(epoch_loader, start=1),
        total=len(epoch_loader),
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    )

    for batch_number, batch in progress:

        if batch_number <= skip:
            continue

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = trial2_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        if not torch.isfinite(loss):
            print("\nNon-finite loss. Saving the previous recovery point.")
            save_checkpoint(epoch, batch_number - 1)
            raise RuntimeError("Training stopped because loss was non-finite.")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            MAX_GRAD_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        global_step += 1
        loss_value = loss.detach().float().item()
        loss_history.append(loss_value)

        recent_losses = loss_history[-100:]
        progress.set_postfix(
            loss=f"{loss_value:.4f}",
            avg100=f"{np.mean(recent_losses):.4f}"
        )

        if global_step % 100 == 0:
            progress.write(
                f"Step {global_step}/{TOTAL_STEPS} "
                f"| loss={loss_value:.4f}"
            )

        if global_step % CHECKPOINT_EVERY == 0:
            save_checkpoint(epoch, batch_number)

    save_checkpoint(epoch + 1, 0)
    batches_done = 0


# Saving the final summary

trial2_model.eval()

summary = {
    "run_name": RUN_NAME,
    "epochs_requested": NUM_EPOCHS,
    "optimizer_steps_completed": global_step,
    "total_steps_expected": TOTAL_STEPS,
    "training_complete": global_step >= TOTAL_STEPS,
    "training_samples": len(qwen_train_model),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "first_100_step_mean_loss": (
        float(np.mean(loss_history[:100])) if loss_history else None
    ),
    "last_100_step_mean_loss": (
        float(np.mean(loss_history[-100:])) if loss_history else None
    ),
    "total_elapsed_seconds": elapsed_before + time.time() - session_start,
}

with open(os.path.join(RUN_DIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\nTraining summary:")
print(json.dumps(summary, indent=2))

GPU: Tesla T4
Training examples: 120000
Batch size: 8
Steps per epoch: 15000
Learning rate: 2e-05
Checkpoint folder: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial2_lr2e5/checkpoints/qwen_lora_trial2


Epoch 1/1:   0%|          | 0/15000 [00:00<?, ?it/s]

Step 100/15000 | loss=2.3626
Step 200/15000 | loss=1.5757
Step 300/15000 | loss=1.4541
Step 400/15000 | loss=1.1881
Step 500/15000 | loss=1.2990

Checkpoint saved: step 500
Step 600/15000 | loss=0.4401
Step 700/15000 | loss=0.4780
Step 800/15000 | loss=0.2870
Step 900/15000 | loss=0.1743
Step 1000/15000 | loss=0.0607

Checkpoint saved: step 1000
Step 1100/15000 | loss=0.1069
Step 1200/15000 | loss=0.5587
Step 1300/15000 | loss=0.1236
Step 1400/15000 | loss=0.1015
Step 1500/15000 | loss=0.5780

Checkpoint saved: step 1500
Step 1600/15000 | loss=0.0265
Step 1700/15000 | loss=0.6494
Step 1800/15000 | loss=0.0037
Step 1900/15000 | loss=0.7300
Step 2000/15000 | loss=0.0212

Checkpoint saved: step 2000
Step 2100/15000 | loss=0.0969
Step 2200/15000 | loss=0.6527
Step 2300/15000 | loss=0.0272
Step 2400/15000 | loss=0.7935
Step 2500/15000 | loss=0.1563

Checkpoint saved: step 2500
Step 2600/15000 | loss=0.0047
Step 2700/15000 | loss=0.5675
Step 2800/15000 | loss=1.1264
Step 2900/15000 | loss=0.

Trial 2 Evaluation

In [45]:

import os
import json
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Use the trained Trial 2 model for evaluation
trial2_model.eval()

all_predictions = []
all_labels = []

total_test_loss = 0.0
total_examples = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Qwen Trial 2"):

        # Move data to GPU
        batch = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        # Get predictions
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = trial2_model(**batch)

        # Calculate test loss
        batch_size = batch["labels"].size(0)
        total_test_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

        # Select the category with the highest score
        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

# Calculate evaluation metrics
test_accuracy = accuracy_score(all_labels, all_predictions)
test_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)
average_test_loss = total_test_loss / total_examples

class_names = ["World", "Sports", "Business", "Sci/Tech"]

print("\n===== QWEN TRIAL 2 TEST RESULTS =====")
print(f"Test examples: {total_examples}")
print(f"Test loss: {average_test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1: {test_macro_f1:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("===== CONFUSION MATRIX =====")
print(confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
))

# Save evaluation results to Google Drive
results_dir = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA/qwen_lora_trial2_lr2e5"
)
os.makedirs(results_dir, exist_ok=True)

results = {
    "trial": 2,
    "model": "Qwen2.5-0.5B-Instruct with LoRA",
    "learning_rate": 2e-5,
    "batch_size": 8,
    "epochs": 1,
    "training_examples": 120000,
    "test_examples": int(total_examples),
    "test_loss": float(average_test_loss),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "classification_report": classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        output_dict=True,
        zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3]
    ).tolist()
}

results_path = os.path.join(
    results_dir, "qwen_test_results.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {results_path}")

Evaluating Qwen Trial 2:   0%|          | 0/950 [00:00<?, ?it/s]


===== QWEN TRIAL 2 TEST RESULTS =====
Test examples: 7600
Test loss: 0.2858
Accuracy: 0.9300 (93.00%)
Macro-F1: 0.9301

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

       World     0.9568    0.9205    0.9383      1900
      Sports     0.9776    0.9874    0.9825      1900
    Business     0.8833    0.9079    0.8954      1900
    Sci/Tech     0.9042    0.9042    0.9042      1900

    accuracy                         0.9300      7600
   macro avg     0.9305    0.9300    0.9301      7600
weighted avg     0.9305    0.9300    0.9301      7600

===== CONFUSION MATRIX =====
[[1749   26   70   55]
 [  11 1876   10    3]
 [  42    9 1725  124]
 [  26    8  148 1718]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial2_lr2e5/qwen_test_results.json


TRIAL 3: Qwen LoRA Training
Learning rate: 1e-5 | Batch size: 8 | Epochs: 2

In [46]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

BASE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA"
)
INITIAL_ADAPTER_DIR = os.path.join(BASE_DIR, "initial_adapter")

assert torch.cuda.is_available(), "GPU is not available."
assert os.path.isdir(INITIAL_ADAPTER_DIR), (
    "Initial adapter folder was not found."
)

trial3_tokenizer = AutoTokenizer.from_pretrained(
    INITIAL_ADAPTER_DIR
)

trial3_base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=4,
    torch_dtype=torch.float32,
    device_map="cpu"
)

trial3_base_model.config.pad_token_id = trial3_tokenizer.pad_token_id

trial3_model = PeftModel.from_pretrained(
    trial3_base_model,
    INITIAL_ADAPTER_DIR,
    is_trainable=True
)

trial3_model.config.pad_token_id = trial3_tokenizer.pad_token_id
trial3_model.to("cuda")

print("Fresh Trial 2 model loaded.")
print("Device:", next(trial3_model.parameters()).device)
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in trial3_model.parameters() if p.requires_grad):,}"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh Trial 2 model loaded.
Device: cuda:0
Trainable parameters: 544,256


Trail 3 Qwen Training

In [47]:


import os
import json
import time
import random
import numpy as np
import torch

from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from tqdm.auto import tqdm
from safetensors.torch import load_file
from peft import set_peft_model_state_dict


# Settings


RUN_NAME = "qwen_lora_trial3"
LEARNING_RATE = 1e-5
NUM_EPOCHS = 2
CHECKPOINT_EVERY = 500
MAX_GRAD_NORM = 1.0
SEED = 42
DEVICE = "cuda"

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA"
)

RUN_DIR = os.path.join(CHECKPOINT_DIR, RUN_NAME)
CHECKPOINTS_DIR = os.path.join(RUN_DIR, "checkpoints")
POINTER_FILE = os.path.join(RUN_DIR, "latest_checkpoint.json")
SUMMARY_FILE = os.path.join(RUN_DIR, "training_summary.json")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)


# Verifying Setup


assert torch.cuda.is_available(), "GPU is not available."
assert "trial3_model" in globals(), "trial3_model is missing."
assert "trial3_tokenizer" in globals(), "trial3_tokenizer is missing."
assert "qwen_train_model" in globals(), "Training dataset is missing."
assert "qwen_train_model_ds" in globals(), "Tokenized training dataset is missing."
assert "train_loader" in globals(), "train_loader is missing."

assert len(qwen_train_model) == 120000, (
    f"Expected 120000 training samples, got {len(qwen_train_model)}."
)

print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(qwen_train_model))


# Calculating Training Steps

BATCH_SIZE = train_loader.batch_size or 8
STEPS_PER_EPOCH = (
    len(qwen_train_model) + BATCH_SIZE - 1
) // BATCH_SIZE

TOTAL_STEPS_EXPECTED = STEPS_PER_EPOCH * NUM_EPOCHS

print("Batch size:", BATCH_SIZE)
print("Steps per epoch:", STEPS_PER_EPOCH)
print("Total expected optimizer steps:", TOTAL_STEPS_EXPECTED)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", NUM_EPOCHS)
print("Checkpoint folder:", CHECKPOINTS_DIR)


# Model and Optimizer


trial3_model.to(DEVICE)

for param in trial3_model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainable_params = [
    param for param in trial3_model.parameters()
    if param.requires_grad
]

assert len(trainable_params) > 0, "No trainable parameters found."

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")


# Restore Previous Progress, If Available


global_step = 0
start_epoch = 0
batches_done = 0
loss_history = []
elapsed_before = 0.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if os.path.isfile(POINTER_FILE):
    with open(POINTER_FILE) as f:
        pointer = json.load(f)

    checkpoint_path = pointer["checkpoint_dir"]
    state_path = os.path.join(
        checkpoint_path, "training_state.pt"
    )
    adapter_path = os.path.join(
        checkpoint_path, "adapter_model.safetensors"
    )

    assert os.path.isfile(state_path), (
        f"Training state is missing: {state_path}"
    )
    assert os.path.isfile(adapter_path), (
        f"Adapter weights are missing: {adapter_path}"
    )

    state = torch.load(
        state_path,
        map_location="cpu",
        weights_only=False
    )

    set_peft_model_state_dict(
        trial3_model,
        load_file(adapter_path)
    )

    optimizer.load_state_dict(state["optimizer_state_dict"])
    scaler.load_state_dict(state["scaler_state_dict"])

    global_step = state["global_step"]
    start_epoch = state["epoch_index"]
    batches_done = state["batches_done"]
    loss_history = state["loss_history"]
    elapsed_before = state["elapsed_seconds"]

    random.setstate(state["python_rng_state"])
    np.random.set_state(state["numpy_rng_state"])
    torch.set_rng_state(state["torch_rng_state"])
    torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])

    print("Resuming from:", checkpoint_path)
    print("Restored optimizer steps:", global_step)
    print("Restored epoch index:", start_epoch + 1)

else:
    print("Starting Trial 3 from its initialized model.")


# Checkpoint Function


def save_checkpoint(epoch_index, completed_batches, tag="periodic"):
    checkpoint_name = (
        f"{tag}_step_{global_step:06d}"
        f"_epoch_{epoch_index + 1:02d}"
    )

    checkpoint_path = os.path.join(
        CHECKPOINTS_DIR, checkpoint_name
    )
    os.makedirs(checkpoint_path, exist_ok=True)

    trial3_model.save_pretrained(checkpoint_path)
    trial3_tokenizer.save_pretrained(checkpoint_path)

    state = {
        "global_step": global_step,
        "epoch_index": epoch_index,
        "batches_done": completed_batches,
        "loss_history": loss_history,
        "elapsed_seconds": elapsed_before + (
            time.time() - training_start_time
        ),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all()
    }

    torch.save(
        state,
        os.path.join(checkpoint_path, "training_state.pt")
    )

    pointer = {
        "global_step": global_step,
        "checkpoint_dir": checkpoint_path,
        "epoch_index": epoch_index,
        "batches_done": completed_batches
    }

    temp_pointer = POINTER_FILE + ".tmp"

    with open(temp_pointer, "w") as f:
        json.dump(pointer, f, indent=2)

    os.replace(temp_pointer, POINTER_FILE)


# Training Loop


training_start_time = time.time()
trial3_model.train()

for epoch_index in range(start_epoch, NUM_EPOCHS):

    # Deterministic shuffle for each epoch
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch_index)

    indices = torch.randperm(
        len(qwen_train_model),
        generator=generator
    ).tolist()

    shuffled_dataset = Subset(qwen_train_model, indices)

    epoch_loader = DataLoader(
        shuffled_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=train_loader.collate_fn
    )

    skip_batches = (
        batches_done if epoch_index == start_epoch else 0
    )

    progress = tqdm(
        enumerate(epoch_loader),
        total=len(epoch_loader),
        desc=f"Trial 3 | Epoch {epoch_index + 1}/{NUM_EPOCHS}"
    )

    for batch_index, batch in progress:

        if batch_index < skip_batches:
            continue

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            outputs = trial3_model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at optimizer step {global_step + 1}"
            )

        loss_history.append(float(loss.detach().cpu()))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            MAX_GRAD_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        global_step += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            step=f"{global_step}/{TOTAL_STEPS_EXPECTED}"
        )

        completed_batches = batch_index + 1

        if global_step % CHECKPOINT_EVERY == 0:
            save_checkpoint(
                epoch_index,
                completed_batches
            )

    batches_done = 0


# Final Checkpoint and Training Summary


save_checkpoint(
    NUM_EPOCHS - 1,
    STEPS_PER_EPOCH,
    tag="final"
)

total_elapsed = elapsed_before + (
    time.time() - training_start_time
)

training_complete = global_step >= TOTAL_STEPS_EXPECTED

summary = {
    "run_name": RUN_NAME,
    "trial": 3,
    "epochs_requested": NUM_EPOCHS,
    "optimizer_steps_completed": global_step,
    "total_steps_expected": TOTAL_STEPS_EXPECTED,
    "training_complete": training_complete,
    "training_samples": len(qwen_train_model),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "first_100_step_mean_loss": (
        float(np.mean(loss_history[:100]))
        if loss_history else None
    ),
    "last_100_step_mean_loss": (
        float(np.mean(loss_history[-100:]))
        if loss_history else None
    ),
    "total_elapsed_seconds": total_elapsed
}

with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("\n===== TRIAL 3 TRAINING SUMMARY =====")
print(json.dumps(summary, indent=2))
print("\nSummary saved to:", SUMMARY_FILE)
print("Latest checkpoint pointer:", POINTER_FILE)

GPU: Tesla T4
Training examples: 120000
Batch size: 8
Steps per epoch: 15000
Total expected optimizer steps: 30000
Learning rate: 1e-05
Epochs: 2
Checkpoint folder: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial3/checkpoints
Starting Trial 3 from its initialized model.


Trial 3 | Epoch 1/2:   0%|          | 0/15000 [00:00<?, ?it/s]

Trial 3 | Epoch 2/2:   0%|          | 0/15000 [00:00<?, ?it/s]


===== TRIAL 3 TRAINING SUMMARY =====
{
  "run_name": "qwen_lora_trial3",
  "trial": 3,
  "epochs_requested": 2,
  "optimizer_steps_completed": 30000,
  "total_steps_expected": 30000,
  "training_complete": true,
  "training_samples": 120000,
  "batch_size": 8,
  "learning_rate": 1e-05,
  "first_100_step_mean_loss": 5.323847124576568,
  "last_100_step_mean_loss": 0.25682972818613053,
  "total_elapsed_seconds": 9518.905596017838
}

Summary saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial3/training_summary.json
Latest checkpoint pointer: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial3/latest_checkpoint.json


In [48]:

import os
import json
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Use the trained Trial 3 model for evaluation
trial3_model.eval()

all_predictions = []
all_labels = []

total_test_loss = 0.0
total_examples = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Qwen Trial 3"):

        # Move data to GPU
        batch = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        # Get predictions
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = trial3_model(**batch)

        # Calculate test loss
        batch_size = batch["labels"].size(0)
        total_test_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

        # Select the category with the highest score
        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

# Calculate evaluation metrics
test_accuracy = accuracy_score(all_labels, all_predictions)
test_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)
average_test_loss = total_test_loss / total_examples

class_names = ["World", "Sports", "Business", "Sci/Tech"]

print("\n===== QWEN TRIAL 3 TEST RESULTS =====")
print(f"Test examples: {total_examples}")
print(f"Test loss: {average_test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1: {test_macro_f1:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("===== CONFUSION MATRIX =====")
print(confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
))

# Save evaluation results to Google Drive
results_dir = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA/qwen_lora_trial3"
)
os.makedirs(results_dir, exist_ok=True)

results = {
    "trial": 3,
    "model": "Qwen2.5-0.5B-Instruct with LoRA",
    "learning_rate": 1e-5,
    "batch_size": 8,
    "epochs": 2,
    "training_examples": 120000,
    "test_examples": int(total_examples),
    "test_loss": float(average_test_loss),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "classification_report": classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        output_dict=True,
        zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3]
    ).tolist()
}

results_path = os.path.join(
    results_dir, "qwen_test_results.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {results_path}")

Evaluating Qwen Trial 3:   0%|          | 0/950 [00:00<?, ?it/s]


===== QWEN TRIAL 3 TEST RESULTS =====
Test examples: 7600
Test loss: 0.2893
Accuracy: 0.9334 (93.34%)
Macro-F1: 0.9334

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

       World     0.9494    0.9374    0.9433      1900
      Sports     0.9827    0.9879    0.9853      1900
    Business     0.8989    0.8984    0.8987      1900
    Sci/Tech     0.9029    0.9100    0.9064      1900

    accuracy                         0.9334      7600
   macro avg     0.9335    0.9334    0.9334      7600
weighted avg     0.9335    0.9334    0.9334      7600

===== CONFUSION MATRIX =====
[[1781   16   59   44]
 [  11 1877    9    3]
 [  45    9 1707  139]
 [  39    8  124 1729]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial3/qwen_test_results.json


Loading the DistilBERT Model

In [14]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

DISTILBERT_MODEL_NAME = "distilbert/distilbert-base-uncased"

distilbert_tokenizer = AutoTokenizer.from_pretrained(
    DISTILBERT_MODEL_NAME
)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=4,
    id2label={
        0: "World",
        1: "Sports",
        2: "Business",
        3: "Sci/Tech"
    },
    label2id={
        "World": 0,
        "Sports": 1,
        "Business": 2,
        "Sci/Tech": 3
    }
)

print("Model:", DISTILBERT_MODEL_NAME)
print("GPU available:", torch.cuda.is_available())
print("Number of labels:", distilbert_model.config.num_labels)
print("Tokenizer vocabulary size:", len(distilbert_tokenizer))
print(
    "Model parameters:",
    f"{sum(p.numel() for p in distilbert_model.parameters()):,}"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert/distilbert-base-uncased
GPU available: True
Number of labels: 4
Tokenizer vocabulary size: 30522
Model parameters: 66,956,548


Tokenizing AG News for DistilBERT

In [15]:

from datasets import DatasetDict

MAX_LENGTH = 256

distilbert_tokenized_ds = DatasetDict({
    split: ag_news_prepared_ds[split].map(
        lambda batch: distilbert_tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH
        ),
        batched=True,
        desc=f"Tokenizing DistilBERT {split} data"
    )
    for split in ["train", "test"]
})

# Keep only model inputs and labels
distilbert_tokenized_ds = distilbert_tokenized_ds.rename_column(
    "label", "labels"
)

print("Train examples:", len(distilbert_tokenized_ds["train"]))
print("Test examples:", len(distilbert_tokenized_ds["test"]))
print("Columns:", distilbert_tokenized_ds["train"].column_names)
print("First example keys:", distilbert_tokenized_ds["train"][0].keys())

Train examples: 120000
Test examples: 7600
Columns: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']
First example keys: dict_keys(['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'])


Evaluating the baseline for DistilBERT

In [16]:

import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

# Dynamic padding and model-compatible input fields
distilbert_collator = DataCollatorWithPadding(
    tokenizer=distilbert_tokenizer,
    return_tensors="pt"
)

distilbert_test_eval_ds = distilbert_tokenized_ds["test"].remove_columns(
    ["text", "token_type_ids"]
)

distilbert_test_loader = DataLoader(
    distilbert_test_eval_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

distilbert_model = distilbert_model.to("cuda")
distilbert_model.eval()

all_predictions = []
all_labels = []
total_eval_loss = 0.0
total_examples = 0

device = next(distilbert_model.parameters()).device

with torch.no_grad():
    for batch in tqdm(distilbert_test_loader, desc="Evaluating DistilBERT baseline"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = distilbert_model(**batch)

        batch_size = batch["labels"].size(0)
        total_eval_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

        predictions = torch.argmax(outputs.logits, dim=-1)
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

distilbert_baseline_loss = total_eval_loss / total_examples
distilbert_baseline_accuracy = accuracy_score(all_labels, all_predictions)
distilbert_baseline_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)

print(f"Baseline test loss: {distilbert_baseline_loss:.4f}")
print(f"Baseline test accuracy: {distilbert_baseline_accuracy:.4f}")
print(f"Baseline test macro-F1: {distilbert_baseline_macro_f1:.4f}")
print("\nPer-class results:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    digits=4,
    zero_division=0
))

Evaluating DistilBERT baseline:   0%|          | 0/238 [00:00<?, ?it/s]

Baseline test loss: 1.3972
Baseline test accuracy: 0.2530
Baseline test macro-F1: 0.1070

Per-class results:
              precision    recall  f1-score   support

       World     0.0000    0.0000    0.0000      1900
      Sports     0.9259    0.0132    0.0259      1900
    Business     0.2515    0.9989    0.4019      1900
    Sci/Tech     0.0000    0.0000    0.0000      1900

    accuracy                         0.2530      7600
   macro avg     0.2944    0.2530    0.1070      7600
weighted avg     0.2944    0.2530    0.1070      7600



Setup DistilBERT model with LoRA

In [18]:

import os
import torch
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

# Trial 1 settings
DISTILBERT_NAME = "distilbert/distilbert-base-uncased"
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial1_lr2e5"
)

os.makedirs(DISTILBERT_OUTPUT_DIR, exist_ok=True)

# Reload a fresh base model; do not wrap the existing PEFT model
distilbert_base_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_NAME,
    num_labels=4,
    id2label={
        0: "World",
        1: "Sports",
        2: "Business",
        3: "Sci/Tech",
    },
    label2id={
        "World": 0,
        "Sports": 1,
        "Business": 2,
        "Sci/Tech": 3,
    },
)

# Apply LoRA once to the fresh base model
distilbert_lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
    bias="none",
)

distilbert_lora_model = get_peft_model(
    distilbert_base_model,
    distilbert_lora_config,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
distilbert_lora_model.to(device)

print("\nDistilBERT LoRA setup ready.")
distilbert_lora_model.print_trainable_parameters()
print("Device:", device)
print("Training examples:", len(distilbert_tokenized_ds["train"]))
print("Test examples:", len(distilbert_tokenized_ds["test"]))
print("Output directory:", DISTILBERT_OUTPUT_DIR)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



DistilBERT LoRA setup ready.
trainable params: 741,124 || all params: 67,697,672 || trainable%: 1.0948
Device: cuda
Training examples: 120000
Test examples: 7600
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial1_lr2e5


Prepare DistilBERT LoRA Trial 1 training

Batch size: 8

Learning rate: 2e-05

Epochs: 1




In [20]:

import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Trial 1 configuration
DISTILBERT_BATCH_SIZE = 8
DISTILBERT_LR = 2e-5
DISTILBERT_EPOCHS = 1
DISTILBERT_WEIGHT_DECAY = 0.01
DISTILBERT_MAX_GRAD_NORM = 1.0
DISTILBERT_CHECKPOINT_EVERY = 500

# Dynamic padding; remove text and DistilBERT-unused token_type_ids
distilbert_train_loader_ds = (
    distilbert_tokenized_ds["train"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_test_loader_ds = (
    distilbert_tokenized_ds["test"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_collator = DataCollatorWithPadding(
    tokenizer=distilbert_tokenizer,
    return_tensors="pt"
)

distilbert_train_loader = DataLoader(
    distilbert_train_loader_ds,
    batch_size=DISTILBERT_BATCH_SIZE,
    shuffle=True,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

distilbert_test_loader = DataLoader(
    distilbert_test_loader_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Optimise trainable LoRA parameters and the saved classification head
distilbert_optimizer = torch.optim.AdamW(
    [p for p in distilbert_lora_model.parameters() if p.requires_grad],
    lr=DISTILBERT_LR,
    weight_decay=DISTILBERT_WEIGHT_DECAY
)

DISTILBERT_TOTAL_STEPS = (
    len(distilbert_train_loader) * DISTILBERT_EPOCHS
)

print("Trial 1 training preparation complete.")
print("Training examples:", len(distilbert_train_loader_ds))
print("Test examples:", len(distilbert_test_loader_ds))
print("Training batches per epoch:", len(distilbert_train_loader))
print("Total training steps:", DISTILBERT_TOTAL_STEPS)
print("Batch size:", DISTILBERT_BATCH_SIZE)
print("Learning rate:", DISTILBERT_LR)
print("Epochs:", DISTILBERT_EPOCHS)
print("Checkpoint interval:", DISTILBERT_CHECKPOINT_EVERY)
print("Device:", next(distilbert_lora_model.parameters()).device)

Trial 1 training preparation complete.
Training examples: 120000
Test examples: 7600
Training batches per epoch: 15000
Total training steps: 15000
Batch size: 8
Learning rate: 2e-05
Epochs: 1
Checkpoint interval: 500
Device: cuda:0


Training DistilBERT + LoRA Trial 1

In [21]:

import os
import json
import time
import torch
from tqdm.auto import tqdm
from peft import get_peft_model_state_dict

# Trial-specific paths
DISTILBERT_CHECKPOINT_DIR = os.path.join(
    DISTILBERT_OUTPUT_DIR, "checkpoints"
)
os.makedirs(DISTILBERT_CHECKPOINT_DIR, exist_ok=True)

DISTILBERT_LOG_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "training_log.json"
)

# Fresh run guard: do not silently overwrite previous training
if os.path.exists(DISTILBERT_LOG_PATH):
    raise FileExistsError(
        f"Training log already exists: {DISTILBERT_LOG_PATH}\n"
        "Stop here and inspect existing progress before starting again."
    )

distilbert_lora_model.train()
distilbert_device = next(distilbert_lora_model.parameters()).device

distilbert_step = 0
distilbert_running_loss = 0.0
distilbert_loss_window = 0.0
distilbert_window_steps = 0
distilbert_training_log = []
distilbert_start_time = time.time()

progress = tqdm(
    total=DISTILBERT_TOTAL_STEPS,
    desc="DistilBERT LoRA Trial 1"
)

for epoch in range(DISTILBERT_EPOCHS):
    for batch in distilbert_train_loader:
        batch = {
            key: value.to(distilbert_device)
            for key, value in batch.items()
        }

        distilbert_optimizer.zero_grad(set_to_none=True)

        outputs = distilbert_lora_model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at step {distilbert_step + 1}: {loss.item()}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            distilbert_lora_model.parameters(),
            DISTILBERT_MAX_GRAD_NORM
        )
        distilbert_optimizer.step()

        distilbert_step += 1
        loss_value = loss.item()
        distilbert_running_loss += loss_value
        distilbert_loss_window += loss_value
        distilbert_window_steps += 1

        progress.update(1)
        progress.set_postfix(loss=f"{loss_value:.4f}")

        # Log training loss every 100 steps
        if distilbert_step % 100 == 0:
            mean_loss = (
                distilbert_loss_window / distilbert_window_steps
            )
            distilbert_training_log.append({
                "step": distilbert_step,
                "epoch": epoch + 1,
                "mean_training_loss_100_steps": mean_loss,
                "elapsed_seconds": time.time() - distilbert_start_time
            })

            with open(DISTILBERT_LOG_PATH, "w") as f:
                json.dump(distilbert_training_log, f, indent=2)

            distilbert_loss_window = 0.0
            distilbert_window_steps = 0

        # Save adapter checkpoint every 500 steps
        if distilbert_step % DISTILBERT_CHECKPOINT_EVERY == 0:
            checkpoint_path = os.path.join(
                DISTILBERT_CHECKPOINT_DIR,
                f"checkpoint-{distilbert_step}"
            )
            os.makedirs(checkpoint_path, exist_ok=True)

            distilbert_lora_model.save_pretrained(checkpoint_path)
            distilbert_tokenizer.save_pretrained(checkpoint_path)

            # Save optimizer state for recovery
            torch.save(
                distilbert_optimizer.state_dict(),
                os.path.join(checkpoint_path, "optimizer.pt")
            )

            with open(os.path.join(checkpoint_path, "progress.json"), "w") as f:
                json.dump({
                    "step": distilbert_step,
                    "epoch": epoch,
                    "average_training_loss": (
                        distilbert_running_loss / distilbert_step
                    )
                }, f, indent=2)

progress.close()

# Save final adapter and tokenizer
distilbert_lora_model.save_pretrained(DISTILBERT_OUTPUT_DIR)
distilbert_tokenizer.save_pretrained(DISTILBERT_OUTPUT_DIR)

distilbert_training_summary = {
    "trial": 1,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "batch_size": DISTILBERT_BATCH_SIZE,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": distilbert_step,
    "average_training_loss": distilbert_running_loss / distilbert_step,
    "elapsed_seconds": time.time() - distilbert_start_time,
    "last_100_step_mean_loss": (
        distilbert_training_log[-1]["mean_training_loss_100_steps"]
        if distilbert_training_log else None
    )
}

with open(
    os.path.join(DISTILBERT_OUTPUT_DIR, "training_summary.json"), "w"
) as f:
    json.dump(distilbert_training_summary, f, indent=2)

print("\nTrial 1 training completed.")
print(json.dumps(distilbert_training_summary, indent=2))
print("Adapter saved to:", DISTILBERT_OUTPUT_DIR)
print("Checkpoints saved to:", DISTILBERT_CHECKPOINT_DIR)

DistilBERT LoRA Trial 1:   0%|          | 0/15000 [00:00<?, ?it/s]


Trial 1 training completed.
{
  "trial": 1,
  "learning_rate": 2e-05,
  "epochs": 1,
  "batch_size": 8,
  "training_examples": 120000,
  "total_steps": 15000,
  "average_training_loss": 0.3147667061840184,
  "elapsed_seconds": 720.4774484634399,
  "last_100_step_mean_loss": 0.2534594989940524
}
Adapter saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial1_lr2e5
Checkpoints saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial1_lr2e5/checkpoints


Evaluate Trial 1

In [22]:

import os
import json
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Evaluate the already-trained DistilBERT LoRA adapter
distilbert_lora_model.eval()
distilbert_device = next(distilbert_lora_model.parameters()).device

distilbert_all_labels = []
distilbert_all_predictions = []
distilbert_total_test_loss = 0.0
distilbert_total_test_examples = 0

with torch.no_grad():
    for batch in tqdm(distilbert_test_loader, desc="Evaluating DistilBERT Trial 1"):
        batch = {
            key: value.to(distilbert_device)
            for key, value in batch.items()
        }

        outputs = distilbert_lora_model(**batch)
        batch_size = batch["labels"].size(0)

        distilbert_total_test_loss += outputs.loss.item() * batch_size
        distilbert_total_test_examples += batch_size

        predictions = torch.argmax(outputs.logits, dim=-1)

        distilbert_all_labels.extend(batch["labels"].cpu().numpy().tolist())
        distilbert_all_predictions.extend(predictions.cpu().numpy().tolist())

distilbert_test_loss = (
    distilbert_total_test_loss / distilbert_total_test_examples
)
distilbert_accuracy = accuracy_score(
    distilbert_all_labels, distilbert_all_predictions
)
distilbert_macro_f1 = f1_score(
    distilbert_all_labels,
    distilbert_all_predictions,
    average="macro",
    zero_division=0
)

distilbert_class_names = ["World", "Sports", "Business", "Sci/Tech"]

distilbert_report = classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=distilbert_class_names,
    output_dict=True,
    zero_division=0
)

distilbert_trial1_results = {
    "model": "DistilBERT + LoRA",
    "trial": 1,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "batch_size": DISTILBERT_BATCH_SIZE,
    "training_examples": len(distilbert_train_loader_ds),
    "test_examples": distilbert_total_test_examples,
    "test_loss": distilbert_test_loss,
    "accuracy": distilbert_accuracy,
    "macro_f1": distilbert_macro_f1,
    "classification_report": distilbert_report,
    "confusion_matrix": confusion_matrix(
        distilbert_all_labels,
        distilbert_all_predictions,
        labels=[0, 1, 2, 3]
    ).tolist()
}

distilbert_results_path = os.path.join(
    DISTILBERT_OUTPUT_DIR, "test_results.json"
)

with open(distilbert_results_path, "w") as f:
    json.dump(distilbert_trial1_results, f, indent=2)

print(f"Test loss: {distilbert_test_loss:.4f}")
print(f"Accuracy: {distilbert_accuracy:.4%}")
print(f"Macro-F1: {distilbert_macro_f1:.4f}")
print("\nPer-class evaluation:")
print(classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=distilbert_class_names,
    zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted):")
print(np.array(distilbert_trial1_results["confusion_matrix"]))
print("\nResults saved to:", distilbert_results_path)

Evaluating DistilBERT Trial 1:   0%|          | 0/238 [00:00<?, ?it/s]

Test loss: 0.2726
Accuracy: 91.3158%
Macro-F1: 0.9130

Per-class evaluation:
              precision    recall  f1-score   support

       World       0.94      0.88      0.91      1900
      Sports       0.96      0.99      0.97      1900
    Business       0.89      0.86      0.87      1900
    Sci/Tech       0.86      0.93      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600

Confusion matrix (rows=true, columns=predicted):
[[1680   58   99   63]
 [   8 1874   14    4]
 [  49   10 1628  213]
 [  46    7   89 1758]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial1_lr2e5/test_results.json


Prepare DistilBERT LoRA Trial 2 training

Batch size: 8

Learning rate: 2e-04

Epochs: 1

In [23]:
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Trial 2 configuration
DISTILBERT_BATCH_SIZE = 8
DISTILBERT_LR = 2e-4
DISTILBERT_EPOCHS = 1
DISTILBERT_WEIGHT_DECAY = 0.01
DISTILBERT_MAX_GRAD_NORM = 1.0
DISTILBERT_CHECKPOINT_EVERY = 500

# Dynamic padding; remove text and DistilBERT-unused token_type_ids
distilbert_train_loader_ds = (
    distilbert_tokenized_ds["train"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_test_loader_ds = (
    distilbert_tokenized_ds["test"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_collator = DataCollatorWithPadding(
    tokenizer=distilbert_tokenizer,
    return_tensors="pt"
)

distilbert_train_loader = DataLoader(
    distilbert_train_loader_ds,
    batch_size=DISTILBERT_BATCH_SIZE,
    shuffle=True,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

distilbert_test_loader = DataLoader(
    distilbert_test_loader_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Optimise trainable LoRA parameters and the saved classification head
distilbert_optimizer = torch.optim.AdamW(
    [p for p in distilbert_lora_model.parameters() if p.requires_grad],
    lr=DISTILBERT_LR,
    weight_decay=DISTILBERT_WEIGHT_DECAY
)

DISTILBERT_TOTAL_STEPS = (
    len(distilbert_train_loader) * DISTILBERT_EPOCHS
)

print("Trial 2 training preparation complete.")
print("Training examples:", len(distilbert_train_loader_ds))
print("Test examples:", len(distilbert_test_loader_ds))
print("Training batches per epoch:", len(distilbert_train_loader))
print("Total training steps:", DISTILBERT_TOTAL_STEPS)
print("Batch size:", DISTILBERT_BATCH_SIZE)
print("Learning rate:", DISTILBERT_LR)
print("Epochs:", DISTILBERT_EPOCHS)
print("Checkpoint interval:", DISTILBERT_CHECKPOINT_EVERY)
print("Device:", next(distilbert_lora_model.parameters()).device)

Trial 2 training preparation complete.
Training examples: 120000
Test examples: 7600
Training batches per epoch: 15000
Total training steps: 15000
Batch size: 8
Learning rate: 0.0002
Epochs: 1
Checkpoint interval: 500
Device: cuda:0


Training DistilBERT + LoRA Trial 2

In [24]:
import os
import json
import time
import torch
from tqdm.auto import tqdm

# Trial 2 output directory
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial2_lr2e4"
)

DISTILBERT_CHECKPOINT_DIR = os.path.join(
    DISTILBERT_OUTPUT_DIR, "checkpoints"
)
os.makedirs(DISTILBERT_CHECKPOINT_DIR, exist_ok=True)

DISTILBERT_LOG_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "training_log.json"
)

# Training log
distilbert_training_log = {
    "trial": 2,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "batch_size": DISTILBERT_BATCH_SIZE,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": DISTILBERT_TOTAL_STEPS,
    "step_losses": [],
    "epoch_losses": [],
    "elapsed_seconds": None,
}

# Train
distilbert_lora_model.train()
distilbert_global_step = 0
distilbert_start_time = time.time()
distilbert_running_loss = 0.0

distilbert_progress = tqdm(
    total=DISTILBERT_TOTAL_STEPS,
    desc="DistilBERT LoRA Trial 2"
)

for epoch in range(DISTILBERT_EPOCHS):
    distilbert_epoch_loss = 0.0

    for batch in distilbert_train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        distilbert_optimizer.zero_grad(set_to_none=True)

        outputs = distilbert_lora_model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise ValueError(
                f"Non-finite loss at step {distilbert_global_step + 1}: "
                f"{loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            distilbert_lora_model.parameters(),
            DISTILBERT_MAX_GRAD_NORM
        )

        distilbert_optimizer.step()

        batch_loss = loss.item()
        distilbert_global_step += 1
        distilbert_epoch_loss += batch_loss
        distilbert_running_loss += batch_loss

        distilbert_training_log["step_losses"].append({
            "step": distilbert_global_step,
            "loss": batch_loss
        })

        distilbert_progress.update(1)
        distilbert_progress.set_postfix(
            loss=f"{batch_loss:.4f}",
            lr=f"{DISTILBERT_LR:.1e}"
        )

        # Save resumable training checkpoint every 500 steps
        if (
            distilbert_global_step
            % DISTILBERT_CHECKPOINT_EVERY == 0
        ):
            distilbert_checkpoint_path = os.path.join(
                DISTILBERT_CHECKPOINT_DIR,
                f"checkpoint-{distilbert_global_step}"
            )
            os.makedirs(distilbert_checkpoint_path, exist_ok=True)

            distilbert_lora_model.save_pretrained(
                distilbert_checkpoint_path
            )
            distilbert_tokenizer.save_pretrained(
                distilbert_checkpoint_path
            )
            torch.save(
                {
                    "optimizer_state_dict":
                        distilbert_optimizer.state_dict(),
                    "global_step": distilbert_global_step,
                    "epoch": epoch,
                    "running_loss": distilbert_running_loss,
                },
                os.path.join(
                    distilbert_checkpoint_path,
                    "training_state.pt"
                )
            )

            with open(DISTILBERT_LOG_PATH, "w") as f:
                json.dump(distilbert_training_log, f, indent=2)

    distilbert_average_epoch_loss = (
        distilbert_epoch_loss / len(distilbert_train_loader)
    )
    distilbert_training_log["epoch_losses"].append(
        distilbert_average_epoch_loss
    )

    print(
        f"Trial 2 | Epoch {epoch + 1}/{DISTILBERT_EPOCHS} "
        f"| Average training loss: "
        f"{distilbert_average_epoch_loss:.4f}"
    )

distilbert_progress.close()

# Save final adapter and tokenizer
distilbert_lora_model.save_pretrained(DISTILBERT_OUTPUT_DIR)
distilbert_tokenizer.save_pretrained(DISTILBERT_OUTPUT_DIR)

distilbert_elapsed_seconds = time.time() - distilbert_start_time

distilbert_training_log["average_training_loss"] = (
    distilbert_running_loss / distilbert_global_step
)
distilbert_training_log["elapsed_seconds"] = (
    distilbert_elapsed_seconds
)
distilbert_training_log["last_100_step_mean_loss"] = (
    sum(
        item["loss"]
        for item in distilbert_training_log["step_losses"][-100:]
    )
    / min(100, len(distilbert_training_log["step_losses"]))
)

with open(DISTILBERT_LOG_PATH, "w") as f:
    json.dump(distilbert_training_log, f, indent=2)

print("\nTrial 2 training complete.")
print(json.dumps({
    "trial": 2,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": distilbert_global_step,
    "average_training_loss":
        distilbert_training_log["average_training_loss"],
    "last_100_step_mean_loss":
        distilbert_training_log["last_100_step_mean_loss"],
    "elapsed_seconds": distilbert_elapsed_seconds,
    "adapter_directory": DISTILBERT_OUTPUT_DIR,
    "training_log": DISTILBERT_LOG_PATH,
}, indent=2))

DistilBERT LoRA Trial 2:   0%|          | 0/15000 [00:00<?, ?it/s]

Trial 2 | Epoch 1/1 | Average training loss: 0.2450

Trial 2 training complete.
{
  "trial": 2,
  "learning_rate": 0.0002,
  "epochs": 1,
  "training_examples": 120000,
  "total_steps": 15000,
  "average_training_loss": 0.24498949335931489,
  "last_100_step_mean_loss": 0.30593760827439836,
  "elapsed_seconds": 713.7465059757233,
  "adapter_directory": "/content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial2_lr2e4",
  "training_log": "/content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial2_lr2e4/training_log.json"
}


Evaluate Trial 2

In [25]:
import os
import json
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Trial 2 evaluation paths
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial2_lr2e4"
)

DISTILBERT_RESULTS_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "test_results.json"
)

distilbert_lora_model.eval()

distilbert_all_predictions = []
distilbert_all_labels = []
distilbert_total_test_loss = 0.0
distilbert_test_batches = 0

with torch.no_grad():
    for batch in tqdm(
        distilbert_test_loader,
        desc="Evaluating DistilBERT Trial 2"
    ):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = distilbert_lora_model(**batch)

        distilbert_total_test_loss += outputs.loss.item()
        distilbert_test_batches += 1

        predictions = torch.argmax(outputs.logits, dim=-1)

        distilbert_all_predictions.extend(
            predictions.cpu().tolist()
        )
        distilbert_all_labels.extend(
            batch["labels"].cpu().tolist()
        )

distilbert_test_loss = (
    distilbert_total_test_loss / distilbert_test_batches
)
distilbert_accuracy = accuracy_score(
    distilbert_all_labels,
    distilbert_all_predictions
)
distilbert_macro_f1 = f1_score(
    distilbert_all_labels,
    distilbert_all_predictions,
    average="macro",
    zero_division=0
)

distilbert_class_report = classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    output_dict=True,
    zero_division=0
)

distilbert_confusion_matrix = confusion_matrix(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3]
)

distilbert_test_results = {
    "trial": 2,
    "model": "DistilBERT LoRA",
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "test_examples": len(distilbert_all_labels),
    "test_loss": distilbert_test_loss,
    "accuracy": distilbert_accuracy,
    "macro_f1": distilbert_macro_f1,
    "classification_report": distilbert_class_report,
    "confusion_matrix": distilbert_confusion_matrix.tolist()
}

with open(DISTILBERT_RESULTS_PATH, "w") as f:
    json.dump(distilbert_test_results, f, indent=2)

print("DistilBERT Trial 2 test results")
print(f"Test loss: {distilbert_test_loss:.4f}")
print(f"Accuracy: {distilbert_accuracy:.4%}")
print(f"Macro-F1: {distilbert_macro_f1:.4f}")

print("\nPer-class classification report:")
print(classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print(distilbert_confusion_matrix)

print("\nResults saved to:", DISTILBERT_RESULTS_PATH)

Evaluating DistilBERT Trial 2:   0%|          | 0/238 [00:00<?, ?it/s]

DistilBERT Trial 2 test results
Test loss: 0.2290
Accuracy: 93.0921%
Macro-F1: 0.9310

Per-class classification report:
              precision    recall  f1-score   support

       World       0.94      0.93      0.94      1900
      Sports       0.98      0.98      0.98      1900
    Business       0.90      0.90      0.90      1900
    Sci/Tech       0.90      0.91      0.91      1900

    accuracy                           0.93      7600
   macro avg       0.93      0.93      0.93      7600
weighted avg       0.93      0.93      0.93      7600

Confusion matrix (rows=true, columns=predicted):
[[1764   21   61   54]
 [  24 1860   10    6]
 [  41   11 1715  133]
 [  41    9  114 1736]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial2_lr2e4/test_results.json


Prepare DistilBERT LoRA Trial 3 training

Batch size: 8

Learning rate: 2e-05

Epochs: 2

In [26]:
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Trial 3 configuration
DISTILBERT_BATCH_SIZE = 8
DISTILBERT_LR = 2e-5
DISTILBERT_EPOCHS = 2
DISTILBERT_WEIGHT_DECAY = 0.01
DISTILBERT_MAX_GRAD_NORM = 1.0
DISTILBERT_CHECKPOINT_EVERY = 500

# Dynamic padding; remove text and DistilBERT-unused token_type_ids
distilbert_train_loader_ds = (
    distilbert_tokenized_ds["train"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_test_loader_ds = (
    distilbert_tokenized_ds["test"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_collator = DataCollatorWithPadding(
    tokenizer=distilbert_tokenizer,
    return_tensors="pt"
)

distilbert_train_loader = DataLoader(
    distilbert_train_loader_ds,
    batch_size=DISTILBERT_BATCH_SIZE,
    shuffle=True,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

distilbert_test_loader = DataLoader(
    distilbert_test_loader_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Optimise trainable LoRA parameters and the saved classification head
distilbert_optimizer = torch.optim.AdamW(
    [p for p in distilbert_lora_model.parameters() if p.requires_grad],
    lr=DISTILBERT_LR,
    weight_decay=DISTILBERT_WEIGHT_DECAY
)

DISTILBERT_TOTAL_STEPS = (
    len(distilbert_train_loader) * DISTILBERT_EPOCHS
)

print("Trial 3 training preparation complete.")
print("Training examples:", len(distilbert_train_loader_ds))
print("Test examples:", len(distilbert_test_loader_ds))
print("Training batches per epoch:", len(distilbert_train_loader))
print("Total training steps:", DISTILBERT_TOTAL_STEPS)
print("Batch size:", DISTILBERT_BATCH_SIZE)
print("Learning rate:", DISTILBERT_LR)
print("Epochs:", DISTILBERT_EPOCHS)
print("Checkpoint interval:", DISTILBERT_CHECKPOINT_EVERY)
print("Device:", next(distilbert_lora_model.parameters()).device)

Trial 3 training preparation complete.
Training examples: 120000
Test examples: 7600
Training batches per epoch: 15000
Total training steps: 30000
Batch size: 8
Learning rate: 2e-05
Epochs: 2
Checkpoint interval: 500
Device: cuda:0


Training DistilBERT + LoRA Trial 3

In [27]:
import os
import json
import time
import torch
from tqdm.auto import tqdm

# Trial 3 output directory
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs"
)

DISTILBERT_CHECKPOINT_DIR = os.path.join(
    DISTILBERT_OUTPUT_DIR, "checkpoints"
)
os.makedirs(DISTILBERT_CHECKPOINT_DIR, exist_ok=True)

DISTILBERT_LOG_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "training_log.json"
)

# Training log
distilbert_training_log = {
    "trial": 3,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "batch_size": DISTILBERT_BATCH_SIZE,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": DISTILBERT_TOTAL_STEPS,
    "step_losses": [],
    "epoch_losses": [],
    "elapsed_seconds": None,
}

# Train
distilbert_lora_model.train()
distilbert_global_step = 0
distilbert_start_time = time.time()
distilbert_running_loss = 0.0

distilbert_progress = tqdm(
    total=DISTILBERT_TOTAL_STEPS,
    desc="DistilBERT LoRA Trial 3"
)

for epoch in range(DISTILBERT_EPOCHS):
    distilbert_epoch_loss = 0.0

    for batch in distilbert_train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        distilbert_optimizer.zero_grad(set_to_none=True)

        outputs = distilbert_lora_model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise ValueError(
                f"Non-finite loss at step {distilbert_global_step + 1}: "
                f"{loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            distilbert_lora_model.parameters(),
            DISTILBERT_MAX_GRAD_NORM
        )

        distilbert_optimizer.step()

        batch_loss = loss.item()
        distilbert_global_step += 1
        distilbert_epoch_loss += batch_loss
        distilbert_running_loss += batch_loss

        distilbert_training_log["step_losses"].append({
            "step": distilbert_global_step,
            "loss": batch_loss
        })

        distilbert_progress.update(1)
        distilbert_progress.set_postfix(
            loss=f"{batch_loss:.4f}",
            lr=f"{DISTILBERT_LR:.1e}"
        )

        # Save checkpoint every 500 steps
        if (
            distilbert_global_step
            % DISTILBERT_CHECKPOINT_EVERY == 0
        ):
            distilbert_checkpoint_path = os.path.join(
                DISTILBERT_CHECKPOINT_DIR,
                f"checkpoint-{distilbert_global_step}"
            )
            os.makedirs(distilbert_checkpoint_path, exist_ok=True)

            distilbert_lora_model.save_pretrained(
                distilbert_checkpoint_path
            )
            distilbert_tokenizer.save_pretrained(
                distilbert_checkpoint_path
            )

            torch.save(
                {
                    "optimizer_state_dict":
                        distilbert_optimizer.state_dict(),
                    "global_step": distilbert_global_step,
                    "epoch": epoch,
                    "running_loss": distilbert_running_loss,
                },
                os.path.join(
                    distilbert_checkpoint_path,
                    "training_state.pt"
                )
            )

            with open(DISTILBERT_LOG_PATH, "w") as f:
                json.dump(distilbert_training_log, f, indent=2)

    distilbert_average_epoch_loss = (
        distilbert_epoch_loss / len(distilbert_train_loader)
    )
    distilbert_training_log["epoch_losses"].append(
        distilbert_average_epoch_loss
    )

    print(
        f"Trial 3 | Epoch {epoch + 1}/{DISTILBERT_EPOCHS} "
        f"| Average training loss: "
        f"{distilbert_average_epoch_loss:.4f}"
    )

distilbert_progress.close()

# Save final adapter and tokenizer
distilbert_lora_model.save_pretrained(DISTILBERT_OUTPUT_DIR)
distilbert_tokenizer.save_pretrained(DISTILBERT_OUTPUT_DIR)

distilbert_elapsed_seconds = time.time() - distilbert_start_time

distilbert_training_log["average_training_loss"] = (
    distilbert_running_loss / distilbert_global_step
)
distilbert_training_log["elapsed_seconds"] = (
    distilbert_elapsed_seconds
)
distilbert_training_log["last_100_step_mean_loss"] = (
    sum(
        item["loss"]
        for item in distilbert_training_log["step_losses"][-100:]
    )
    / min(100, len(distilbert_training_log["step_losses"]))
)

with open(DISTILBERT_LOG_PATH, "w") as f:
    json.dump(distilbert_training_log, f, indent=2)

print("\nTrial 3 training complete.")
print(json.dumps({
    "trial": 3,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": distilbert_global_step,
    "average_training_loss":
        distilbert_training_log["average_training_loss"],
    "last_100_step_mean_loss":
        distilbert_training_log["last_100_step_mean_loss"],
    "elapsed_seconds": distilbert_elapsed_seconds,
    "adapter_directory": DISTILBERT_OUTPUT_DIR,
    "training_log": DISTILBERT_LOG_PATH,
}, indent=2))

DistilBERT LoRA Trial 3:   0%|          | 0/30000 [00:00<?, ?it/s]

Trial 3 | Epoch 1/2 | Average training loss: 0.1855
Trial 3 | Epoch 2/2 | Average training loss: 0.1777

Trial 3 training complete.
{
  "trial": 3,
  "learning_rate": 2e-05,
  "epochs": 2,
  "training_examples": 120000,
  "total_steps": 30000,
  "average_training_loss": 0.18159635504507485,
  "last_100_step_mean_loss": 0.19542416166441398,
  "elapsed_seconds": 1484.6532125473022,
  "adapter_directory": "/content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs",
  "training_log": "/content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs/training_log.json"
}


Evaluating Trial 3

In [28]:
import os
import json
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Trial 3 evaluation paths
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs"
)

DISTILBERT_RESULTS_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "test_results.json"
)

distilbert_lora_model.eval()

distilbert_all_predictions = []
distilbert_all_labels = []
distilbert_total_test_loss = 0.0
distilbert_test_batches = 0

with torch.no_grad():
    for batch in tqdm(
        distilbert_test_loader,
        desc="Evaluating DistilBERT Trial 3"
    ):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = distilbert_lora_model(**batch)

        distilbert_total_test_loss += outputs.loss.item()
        distilbert_test_batches += 1

        predictions = torch.argmax(outputs.logits, dim=-1)

        distilbert_all_predictions.extend(
            predictions.cpu().tolist()
        )
        distilbert_all_labels.extend(
            batch["labels"].cpu().tolist()
        )

distilbert_test_loss = (
    distilbert_total_test_loss / distilbert_test_batches
)
distilbert_accuracy = accuracy_score(
    distilbert_all_labels,
    distilbert_all_predictions
)
distilbert_macro_f1 = f1_score(
    distilbert_all_labels,
    distilbert_all_predictions,
    average="macro",
    zero_division=0
)

distilbert_class_report = classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    output_dict=True,
    zero_division=0
)

distilbert_confusion_matrix = confusion_matrix(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3]
)

distilbert_test_results = {
    "trial": 3,
    "model": "DistilBERT LoRA",
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "test_examples": len(distilbert_all_labels),
    "test_loss": distilbert_test_loss,
    "accuracy": distilbert_accuracy,
    "macro_f1": distilbert_macro_f1,
    "classification_report": distilbert_class_report,
    "confusion_matrix": distilbert_confusion_matrix.tolist()
}

with open(DISTILBERT_RESULTS_PATH, "w") as f:
    json.dump(distilbert_test_results, f, indent=2)

print("DistilBERT Trial 3 test results")
print(f"Test loss: {distilbert_test_loss:.4f}")
print(f"Accuracy: {distilbert_accuracy:.4%}")
print(f"Macro-F1: {distilbert_macro_f1:.4f}")

print("\nPer-class classification report:")
print(classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print(distilbert_confusion_matrix)

print("\nResults saved to:", DISTILBERT_RESULTS_PATH)

Evaluating DistilBERT Trial 3:   0%|          | 0/238 [00:00<?, ?it/s]

DistilBERT Trial 3 test results
Test loss: 0.2044
Accuracy: 94.2237%
Macro-F1: 0.9422

Per-class classification report:
              precision    recall  f1-score   support

       World       0.96      0.94      0.95      1900
      Sports       0.98      0.99      0.99      1900
    Business       0.92      0.90      0.91      1900
    Sci/Tech       0.90      0.94      0.92      1900

    accuracy                           0.94      7600
   macro avg       0.94      0.94      0.94      7600
weighted avg       0.94      0.94      0.94      7600

Confusion matrix (rows=true, columns=predicted):
[[1795   11   52   42]
 [  11 1876    7    6]
 [  38    9 1710  143]
 [  30   10   80 1780]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs/test_results.json
